In [25]:
import socket
import os
from pathlib import Path
import csv
from datetime import datetime
import logging

if "__file__" in globals():
    BASE_DIR = Path(__file__).resolve().parent
else:
    BASE_DIR = Path(os.getcwd())

NUT_HOST = "172.21.0.1"
NUT_PORT = 3493
UPS_NAME = "cyberpower"

def nut_command(cmd):
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        s.settimeout(5)
        s.connect((NUT_HOST, NUT_PORT))
        s.sendall((cmd + "\n").encode())
        response = b""
        while True:
            chunk = s.recv(4096)
            if not chunk:
                break
            response += chunk
            if b"END LIST" in response or b"ERR" in response:
                break
        return response.decode()

def get_ups_vars():
    raw = nut_command(f"LIST VAR {UPS_NAME}")
    data = {}
    for line in raw.splitlines():
        if line.startswith("VAR"):
            parts = line.split(" ", 3)
            if len(parts) == 4:
                key = parts[2]
                value = parts[3].strip('"')
                data[key] = value
    return data

# 1. Obtenemos el diccionario con todos los datos
ups_data = get_ups_vars()

# 2. Extraemos las variables con un valor por defecto (0) en caso de que no existan
# Usamos int() para convertir el string que devuelve el comando a un número entero
try:
    battery_charge = int(ups_data.get('battery.charge', 0))
    ups_load = int(ups_data.get('ups.load', 0))
except (ValueError, TypeError):
    # En caso de que el dato no sea un número válido
    battery_charge = 0
    ups_load = 0

ups_watts = round(ups_load * 540 / 100, 1)

In [26]:
stats_file = BASE_DIR / "ups.csv"
stats_file.parent.mkdir(parents=True, exist_ok=True)
file_exists = stats_file.exists()

with open(stats_file, "a", newline="") as f:
    writer = csv.writer(f)
    if not file_exists:
        writer.writerow(["date", "battery_charge", "ups_watts"])
    writer.writerow([
        datetime.now().strftime("%Y-%m-%d %H:%M"),
        battery_charge,
        ups_watts
    ])

msg = f"{datetime.now().strftime('%Y-%m-%d %H:%M:%S')} - ups battery: {battery_charge}, instant power: {ups_watts}W"
logging.info(msg)
print(msg)

2026-04-11 15:22:28 - ups battery: 100, instant power: 91.8W


In [29]:
import pandas as pd
df = pd.read_csv(BASE_DIR / 'ups.csv')
df['date'] = pd.to_datetime(df['date'])
today = pd.Timestamp.now().normalize()
chart_start = today - pd.Timedelta(days=1)

df_filtered = df[(df['date'] >= chart_start)].copy()

ups_data = []
for _, row in df_filtered.iterrows():
    battery_charge = row['battery_charge']
    power = row['power']
    if pd.notna(battery_charge) and battery_charge > 0:
        ups_data.append({
                "date": str(row['date'].date()),
                "battery_charge": round(battery_charge, 1),
                "power": round(power, 1)
            })
ups_data.sort(key=lambda x: x['date'])

labels = [d['date'] for d in ups_data]
battery_charge = [d['battery_charge'] for d in ups_data]
power = [d['power'] for d in ups_data]

print(ups_data)

[{'date': '2026-04-11', 'battery_charge': 100, 'power': 97.2}, {'date': '2026-04-11', 'battery_charge': 100, 'power': 91.8}, {'date': '2026-04-11', 'battery_charge': 100, 'power': 91.8}, {'date': '2026-04-11', 'battery_charge': 100, 'power': 91.8}, {'date': '2026-04-11', 'battery_charge': 100, 'power': 91.8}, {'date': '2026-04-11', 'battery_charge': 100, 'power': 91.8}, {'date': '2026-04-11', 'battery_charge': 100, 'power': 91.8}, {'date': '2026-04-11', 'battery_charge': 100, 'power': 91.8}, {'date': '2026-04-11', 'battery_charge': 100, 'power': 113.4}, {'date': '2026-04-11', 'battery_charge': 100, 'power': 91.8}, {'date': '2026-04-11', 'battery_charge': 100, 'power': 91.8}]
